# ラボ 1b: 出生体重（babyweight）データセットの準備

**学習目標**

1. 環境のセットアップ
1. 出生（natality）データセットの前処理
1. 出生データセットの拡張（オーグメンテーション）
1. BigQuery での train および eval テーブルの作成
1. BigQuery から GCS への CSV 形式でのデータのエクスポート


## はじめに
このノートブックでは、出産前の赤ちゃんの体重を予測するモデルの開発とトレーニングに向けて、babyweight データセットを準備します。BigQuery を使用してデータの拡張と前処理を実行し、それらを AutoML Tables、BigQuery ML、および Cloud AI Platform でトレーニングされる Keras モデルに使用します。

このラボでは、環境のセットアップ、プロジェクト データセットの作成、出生データセットの前処理と拡張、BigQuery での train および eval テーブルの作成、そして BigQuery から GCS への CSV 形式でのデータのエクスポートを行います。

各学習目標は受講者用ラボ ノートブックの __#TODO__ に対応しています。まずそちらのノートブックを完了してから、この [解答ノートブック](../solutions/1b_prepare_data_babyweight.ipynb) を確認してください。

## 環境変数の設定と必要なライブラリの読み込み

必要なライブラリをインポートします。

In [ ]:
import os

from google.cloud import bigquery
%load_ext google.cloud.bigquery

ラボ全体で使用できるように環境変数を設定します。バケット名にはプロジェクト ID を使用します。

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

## ソース データセット

データセットは [BigQuery](https://cloud.google.com/bigquery/) でホストされています。CDC の出生（Natality）データには 1969 年から 2008 年までの米国の出生に関する詳細が含まれており、GCP アカウントを持つ誰でもアクセスできる一般公開データセットです。データセットにアクセスするには [ここ](https://console.cloud.google.com/bigquery?project=bigquery-public-data&p=publicdata&d=samples&t=natality&page=table) をクリックしてください。

出生データセットは約 1 億 3,800 万行、31 列と比較的規模が大きいですが、理解しやすい内容です。`weight_pounds`（ポンド単位の体重）がターゲットであり、モデルが予測するようにトレーニングする連続値です。

## BigQuery データセットと Google Cloud Storage バケットの作成

BigQuery データセットは、テーブル、ビュー、および BigQuery ML で構築されたモデル用のコンテナです。以前のラボでまだ作成していない場合は、__babyweight__ という名前のデータセットを作成しましょう。プロジェクト用の GCS バケットについても同様に行います。

In [ ]:
%%bash

# babyweight 用の BigQuery データセットが存在しない場合は作成
datasetexists=$(bq ls -d | grep -w babyweight)

if [ -n "$datasetexists" ]; then
    echo -e "BigQuery dataset already exists, let's not recreate it."

else
    echo "Creating BigQuery dataset titled: babyweight"
    
    bq --location=US mk --dataset \
        --description "Babyweight" \
        $PROJECT:babyweight
    echo "Here are your current datasets:"
    bq ls
fi
    
## GCS バケットがまだ存在しない場合は作成...
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
    
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "Here are your current buckets:"
    gcloud storage ls
fi

## トレーニングおよび評価データ テーブルの作成

すでに一般公開されているデータセットがあるため、この未加工の入力データを使用してトレーニングおよび評価データ テーブルを作成できます。まず、列を `weight_pounds`、`is_male`、`mother_age`、`plurality`、`gestation_weeks` に限定したデータのサブセットを作成し、いくつかの単純なフィルタリングと、再現可能な分割のためにハッシュ化する列を追加します。

* 注: 以下のテーブル作成コード内のデータセットは、先ほど作成したもの（例: "babyweight"）です。

### データセットの前処理とフィルタリング

トレーニングに適した形式にするために、データに対して実行したい前処理とフィルタリングがいくつかあります。

前処理:
* `is_male` を `BOOL` から `STRING` にキャスト
* `plurality` を `INTEGER` から `STRING` にキャストし、`[1, 2, 3, 4, 5]` を `["Single(1)", "Twins(2)", "Triplets(3)", "Quadruplets(4)", "Quintuplets(5)"]` に変換
* `year` と `month` をハッシュ化した `hashcolumn` を追加

フィルタリング:
* `2000` 年以降のデータのみを対象とする
* `0` より大きい赤ちゃんの体重のみを対象とする
* 年齢が `0` より大きい母親のみを対象とする
* 胎児数（plurality）が `0` より大きいもののみを対象とする
* 妊娠期間（gestation_weeks）が `0` より大きいもののみを対象とする

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE
    babyweight.babyweight_data AS
SELECT
    weight_pounds,
    CAST(is_male AS STRING) AS is_male,
    mother_age,
    CASE
        WHEN plurality = 1 THEN "Single(1)"
        WHEN plurality = 2 THEN "Twins(2)"
        WHEN plurality = 3 THEN "Triplets(3)"
        WHEN plurality = 4 THEN "Quadruplets(4)"
        WHEN plurality = 5 THEN "Quintuplets(5)"
    END AS plurality,
    gestation_weeks,
    FARM_FINGERPRINT(
        CONCAT(
            CAST(year AS STRING),
            CAST(month AS STRING)
        )
    ) AS hashmonth
FROM
    publicdata.samples.natality
WHERE
    year > 2000
    AND weight_pounds > 0
    AND mother_age > 0
    AND plurality > 0
    AND gestation_weeks > 0

### 欠損データをシミュレートするためのデータセットの拡張（オーグメンテーション）

すべての性別情報を `Unknown` に設定し、単胎以外のすべての出産の胎児数を `Multiple(2+)` に設定することで、シミュレートされた babyweight データでデータセットを拡張します。

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE
    babyweight.babyweight_augmented_data AS
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    hashmonth
FROM
    babyweight.babyweight_data
UNION ALL
SELECT
    weight_pounds,
    "Unknown" AS is_male,
    mother_age,
    CASE
        WHEN plurality = "Single(1)" THEN plurality
        ELSE "Multiple(2+)"
    END AS plurality,
    gestation_weeks,
    hashmonth
FROM
    babyweight.babyweight_data

### 拡張データセットの train セットと eval セットへの分割

`hashmonth` を使用して剰余演算（modulo）を適用し、およそ 75/25 の train/eval 分割を取得します。

#### 拡張データセットを train データセットに分割

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE
    babyweight.babyweight_data_train AS
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks
FROM
    babyweight.babyweight_augmented_data
WHERE
    ABS(MOD(hashmonth, 4)) < 3

#### 拡張データセットを eval データセットに分割

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE
    babyweight.babyweight_data_eval AS
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks
FROM
    babyweight.babyweight_augmented_data
WHERE
    ABS(MOD(hashmonth, 4)) = 3

## テーブル作成の確認

データセットとトレーニング データ テーブルが作成されたことを確認します。

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリです。これによりテーブルが存在することを確認できます。
SELECT * FROM babyweight.babyweight_data_train
LIMIT 0

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリです。これによりテーブルが存在することを確認できます。
SELECT * FROM babyweight.babyweight_data_eval
LIMIT 0

## BigQuery から GCS への CSV 形式でのエクスポート

BigQuery Python API を使用して、後の TensorFlow / Keras トレーニングで使用できるように、train テーブルと eval テーブルを CSV 形式で Google Cloud Storage にエクスポートします。上記で使用してきたデータセットを使用し、トレーニング データと評価データの両方に対してプロセスを繰り返します。

In [ ]:
# BigQuery クライアント オブジェクトを構築
client = bigquery.Client()

dataset_name = "babyweight"

# データセット参照オブジェクトを作成
dataset_ref = client.dataset(dataset_id=dataset_name, project=client.project)

# train テーブルと eval テーブルの両方をエクスポート
for step in ["train", "eval"]:
    destination_uri = os.path.join(
        "gs://", BUCKET, dataset_name, "data", f"{step}*.csv"
    )
    table_name = f"babyweight_data_{step}"
    table_ref = dataset_ref.table(table_name)
    extract_job = client.extract_table(
        table_ref,
        destination_uri,
        # ロケーションはソース テーブルのロケーションと一致する必要があります
        location="US",
    )  # API リクエスト
    extract_job.result()  # ジョブが完了するまで待機

    print(
        "Exported {}:{}.{} to {}".format(
            client.project, dataset_name, table_name, destination_uri
        )
    )

## CSV 作成の確認

バケット内に CSV ファイルが正しく作成されたことを確認します。

In [ ]:
%%bash
gcloud storage ls gs://${BUCKET}/babyweight/data/*.csv

In [ ]:
%%bash
gcloud storage cat gs://${BUCKET}/babyweight/data/train000000000000.csv 2>/dev/null | head -5

In [ ]:
%%bash
gcloud storage cat gs://${BUCKET}/babyweight/data/eval000000000000.csv 2>/dev/null | head -5

## ラボのまとめ:
このラボでは、環境のセットアップ、BigQuery データセットの作成、出生データセットの前処理と拡張、BigQuery での train および eval テーブルの作成、そして BigQuery から GCS への CSV 形式でのデータのエクスポートを行いました。

Copyright 2021 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.